# Lesson 9 · Ideal gases and kinetic theory

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/StativeSh/quantum-playground/blob/main/lessons/09_ideal_gases_and_kinetic_theory.ipynb)

**For:** A-level Physics (for example AQA 3.6.2.2 to 3.6.2.4, OCR A 5.1.4, Edexcel Topic 9) and IB Physics (B.3 gas laws). Check your own specification for the exact wording.

**Before this lesson:** Lesson 3, *Circular motion and gravitational orbits* (force as rate of change of momentum and kinetic energy). Also assumed: momentum, kinetic energy and the kelvin scale.

**Leads to:** Lesson 16, *Thermal energy transfer*.

**You will be able to:**

1. use the gas laws and the ideal gas equation $pV = nRT = Nk_BT$, with temperature in kelvin
2. describe the kinetic theory model and explain how molecular collisions produce pressure, $pV = \tfrac13Nm\overline{c^2}$
3. use $\tfrac12m\overline{c^2} = \tfrac32k_BT$ and calculate r.m.s. speeds
4. describe the Maxwell–Boltzmann distribution and how it changes with temperature and molecular mass
5. state the assumptions of kinetic theory and explain when a real gas stops being ideal

**How to use this lesson:** run each cell with Shift+Enter. Where you see *Try it*, change a number and run the cell again. Before some experiments a **Predict** box asks you to commit to an answer; write it down, run the cell, then answer the **Explain** box below it. Answers to both are at the very end, after the worked solutions. Exercises are at the end, with worked solutions below them.

In [ ]:
# Setup: installs the qplay package if it is missing (for example on Google Colab).
import importlib.util, subprocess, sys
if importlib.util.find_spec("qplay") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "quantum-playground"], check=True)

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from qplay import kinetic

k_B = 1.381e-23       # Boltzmann constant, J/K
N_A = 6.022e23        # Avogadro constant, 1/mol
R = k_B * N_A         # molar gas constant, about 8.31 J/(mol K)
u = 1.661e-27         # atomic mass unit, kg

## 1. The ideal gas equation

Three experimental laws describe a fixed mass of gas:

- **Boyle's law** (constant temperature): $pV = $ constant
- **Charles's law** (constant pressure): $V \propto T$
- **the pressure law** (constant volume): $p \propto T$

with $T$ in kelvin. They combine into the **ideal gas equation**

$$pV = nRT = Nk_BT,$$

where $n$ is the number of moles, $N = nN_A$ the number of molecules, $R = 8.31\ \mathrm{J\,mol^{-1}\,K^{-1}}$ and $k_B = R/N_A = 1.38\times10^{-23}\ \mathrm{J\,K^{-1}}$.

In [ ]:
n = 1.0                                       # mol
V = np.linspace(0.005, 0.05, 300)             # m^3
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.8))
for T in [200, 300, 400, 600]:
    ax1.plot(V * 1000, n * R * T / V / 1000, label=f"{T} K")
ax1.set(xlabel="V / litres", ylabel="p / kPa", title="Isotherms: each is pV = constant"); ax1.legend()
T = np.linspace(0, 600, 100)
for V_fixed in [0.01, 0.02, 0.04]:
    ax2.plot(T, n * R * T / V_fixed / 1000, label=f"V = {V_fixed * 1000:.0f} L")
ax2.set(xlabel="T / K", ylabel="p / kPa", title="Pressure law: straight lines through 0 K"); ax2.legend();

Every pressure–temperature line, extended backwards, reaches zero pressure at the same temperature: **absolute zero**, 0 K = −273.15 °C. That is how the kelvin scale was found.

## 2. A gas made of molecules

Why does a gas obey such a simple law? **Kinetic theory** pictures it as a huge number of tiny molecules flying about at random and bouncing elastically off each other and the walls. The package simulates exactly that: hard spheres in a cubic box. We start every molecule at the **same speed** in a random direction, and watch.

(The simulation uses its own units, with $k_B = 1$ and molecular mass 1, so "temperature" is an energy.)

**Predict (P1).** Every molecule starts at the same speed. After thousands of collisions, will they still all have the same speed? If not, sketch the shape you expect the distribution of speeds to take. Write your prediction down before you run the next cell.

In [ ]:
gas = kinetic.gas_box(n=400, dim=3, radius=0.02, t_max=6, save_every=20, seed=1)   # about 1 s to run
kT = gas.temperature()
speeds = np.linalg.norm(gas.velocities, axis=-1)          # shape (saved times, molecules)
v = np.linspace(0, 2.5, 300)
fig, axes = plt.subplots(1, 4, figsize=(14, 3.2), sharey=True)
for ax, k in zip(axes, [0, 2, 10, None]):
    data = gas.speeds(len(gas.t) // 2) if k is None else speeds[k]
    ax.hist(data, bins=np.linspace(0, 2.5, 31), density=True, alpha=0.6)
    ax.plot(v, kinetic.maxwell_boltzmann_speed(v, kT), "k", lw=1.2)
    ax.set(xlabel="speed", title="averaged over t > 3" if k is None else f"t = {gas.t[k]:.2f}")
axes[0].set_ylabel("fraction per unit speed")
print(f"{gas.n_particles} molecules, {gas.collisions[-1]} collisions; total kinetic energy changed by "
      f"{np.ptp((gas.velocities**2).sum(axis=(1, 2))) / (gas.velocities[0]**2).sum():.1e} (relative)")

**Explain (P1).** The total kinetic energy never changes. How can the spread of speeds change anyway?

Collisions quickly spread the speeds out. Some molecules are slowed almost to rest; a few are sped up to more than twice the starting speed. The total energy never changes, yet the speeds always settle into the same shape, the **Maxwell–Boltzmann distribution** (black line):

$$f(v) = 4\pi v^2\left(\frac{m}{2\pi k_BT}\right)^{3/2}e^{-mv^2/2k_BT}.$$

Nothing in the simulation "knows" this formula; it comes out of random elastic collisions alone.

## 3. Pressure from molecular collisions

Each time a molecule bounces off a wall it reverses its momentum, pushing on the wall. Adding up all those pushes gives the pressure. The A-level derivation (a molecule bouncing between the walls of a cube, then averaging over directions) gives

$$pV = \tfrac13 Nm\overline{c^2},$$

where $\overline{c^2}$ is the mean square speed. The simulation records every impulse on the walls, so we can measure the pressure directly and compare.

**Predict (P2).** Will the pressure measured from the wall collisions be exactly $\tfrac13Nm\overline{c^2}/V$, a little more, or a little less? Write your prediction down before you run the next cell.

In [ ]:
p_measured = gas.pressure()
mean_c2 = (gas.velocities[0] ** 2).sum(axis=-1).mean()       # constant, since collisions are elastic
p_theory = gas.n_particles * gas.m * mean_c2 / (3 * gas.volume)
print(f"pressure from wall impulses:   {p_measured:.2f}")
print(f"(1/3) N m <c^2> / V:           {p_theory:.2f}")
print(f"ratio:                          {p_measured / p_theory:.3f}")
print(f"molecules fill {100 * gas.packing_fraction:.2f}% of the box; hard-sphere theory predicts a ratio of 1 + 4*eta = {1 + 4 * gas.packing_fraction:.3f}")

**Explain (P2).** The molecules here are not points. How does their size affect how often they hit the walls?

The measured pressure agrees with $\tfrac13 Nm\overline{c^2}/V$ to about 5%, and the small excess is itself predicted: real molecules take up some space, so they hit the walls slightly more often (section 5).

## 4. What temperature means

Put $pV = \tfrac13Nm\overline{c^2}$ next to $pV = Nk_BT$ and you get

$$\tfrac12 m\overline{c^2} = \tfrac32 k_BT .$$

**Temperature measures the average kinetic energy of the molecules.** At a given temperature every gas has the same average kinetic energy per molecule, so lighter molecules move faster: $c_{rms} = \sqrt{\overline{c^2}} = \sqrt{3k_BT/m}$.

**Predict (P3).** At 300 K, how many times faster (r.m.s.) is a helium atom (4.0 u) than a nitrogen molecule (28.0 u)? Write your prediction down before you run the next cell.

In [ ]:
v = np.linspace(0, 3000, 600)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.8))
m_N2 = 28.0 * u
for T in [100, 300, 1000]:
    ax1.plot(v, kinetic.maxwell_boltzmann_speed(v, k_B * T, m_N2), label=f"N$_2$ at {T} K")
ax1.set(xlabel="speed / m s$^{-1}$", ylabel="fraction per unit speed", title="Hotter: faster and more spread out"); ax1.legend()
print(f"{'gas':6s} {'c_rms at 300 K / m s^-1':>24s}")
for i, (name, mass_u) in enumerate([("H2", 2.016), ("He", 4.003), ("N2", 28.0), ("CO2", 44.0)]):
    m = mass_u * u
    ax2.plot(v, kinetic.maxwell_boltzmann_speed(v, k_B * 300, m), label=name)
    print(f"{name:6s} {kinetic.rms_speed(k_B * 300, m):24.0f}")
ax2.set(xlabel="speed / m s$^{-1}$", title="All at 300 K: lighter is faster"); ax2.legend();

**Explain (P3).** Why does the answer depend on the square root of the mass ratio?

Hydrogen and helium molecules move several times faster than nitrogen. Some in the fast tail exceed the Earth's escape velocity (11.2 km s$^{-1}$) high in the atmosphere, which is why the Earth has lost almost all its hydrogen and helium.

## 5. When is a gas "ideal"?

Kinetic theory assumes, among other things, that

- the molecules themselves take up a negligible fraction of the volume
- there are no forces between molecules except during collisions
- collisions are elastic and last a negligible time.

The simulation has no attractive forces, but we can make the molecules bigger and see the first assumption fail. For hard spheres filling a fraction $\eta$ of the box, the Carnahan–Starling equation

$$\frac{pV}{Nk_BT} = \frac{1 + \eta + \eta^2 - \eta^3}{(1 - \eta)^3} \approx 1 + 4\eta + 10\eta^2 + \dots$$

is accurate up to quite dense fluids.

In [ ]:
def carnahan_starling(eta):
    return (1 + eta + eta**2 - eta**3) / (1 - eta) ** 3

rows = []
for radius in [0.005, 0.02, 0.04, 0.06]:                     # <- Try it: other sizes (bigger is slower to set up)
    g = kinetic.gas_box(n=300, dim=3, radius=radius, t_max=3, save_every=50, seed=2)
    Z = g.pressure() * g.volume / (g.n_particles * g.temperature())
    rows.append((g.packing_fraction, Z))
    print(f"radius {radius:5.3f}: fills {100 * g.packing_fraction:5.2f}% of the box,  pV/NkT = {Z:.3f},  Carnahan-Starling {carnahan_starling(g.packing_fraction):.3f}")
eta = np.linspace(0, 0.3, 100)
plt.figure(figsize=(6, 3.6))
plt.plot(eta, np.ones_like(eta), "k:", label="ideal gas")
plt.plot(eta, carnahan_starling(eta), "k--", label="Carnahan-Starling")
plt.plot(*np.array(rows).T, "o", label="simulation")
plt.xlabel(r"fraction of volume filled, $\eta$"); plt.ylabel(r"$pV / Nk_BT$"); plt.legend();

When the molecules fill less than about 1% of the space the gas is ideal to within a few per cent. Air at room temperature is about 0.1% molecules by volume, so it behaves very nearly ideally. Squeezed to high pressure, or cooled until attractions matter (near liquefying), real gases deviate.

---

## Exercises

Take $R = 8.31\ \mathrm{J\,mol^{-1}\,K^{-1}}$, $k_B = 1.38\times10^{-23}\ \mathrm{J\,K^{-1}}$, $N_A = 6.02\times10^{23}\ \mathrm{mol^{-1}}$ and $1\ \mathrm{u} = 1.66\times10^{-27}$ kg. Give answers to 3 significant figures.

**Q1.** A cylinder holds $0.020\ \mathrm{m^3}$ of gas at $1.0\times10^{5}$ Pa and 300 K. Find (a) the number of moles, (b) the number of molecules.

**Q2.** The gas in Q1 is heated to 450 K at constant volume. Find the new pressure.

**Q3.** Find the root mean square speed of nitrogen molecules (mass 28.0 u) at 300 K.

**Q4.** Find the mean kinetic energy of a gas molecule at 27 °C.

**Q5.** At what temperature would the r.m.s. speed of helium atoms (4.00 u) equal the Earth's escape velocity, $11.2\ \mathrm{km\,s^{-1}}$?

**Q6 (simulation).** Predict $pV/Nk_BT$ for 200 molecules of radius 0.03 in the unit box using $1 + 4\eta + 10\eta^2$, where $\eta = N\cdot\frac43\pi r^3$. Then run `kinetic.gas_box(n=200, radius=0.03, t_max=4, seed=5)` and compare.

In [ ]:
def check(name, yours, correct, tol=0.01):
    if yours is None:
        print(f"{name}: not answered yet")
    elif abs(yours - correct) <= tol * abs(correct):
        print(f"{name}: correct")
    else:
        print(f"{name}: not yet, check your working (you gave {yours})")

# The correct values are computed here from the physics, not typed in.
_n1 = 1.0e5 * 0.020 / (R * 300)
_eta6 = 200 * 4 / 3 * np.pi * 0.03**3
_answers = {
    "Q1a": _n1, "Q1b": _n1 * N_A,
    "Q2": 1.0e5 * 450 / 300,
    "Q3": kinetic.rms_speed(k_B * 300, 28.0 * u),
    "Q4": 1.5 * k_B * (27 + 273),
    "Q5": 4.00 * u * 11.2e3**2 / (3 * k_B),
    "Q6": 1 + 4 * _eta6 + 10 * _eta6**2,
}

# Replace None with your answers, in SI units.
mine = {"Q1a": None, "Q1b": None, "Q2": None, "Q3": None, "Q4": None, "Q5": None, "Q6": None}
for q in mine:
    check(q, mine[q], _answers[q])

---

## Worked solutions

Try the questions before reading on.

**Q1.** (a) $n = \dfrac{pV}{RT} = \dfrac{1.0\times10^5 \times 0.020}{8.31 \times 300} = 0.802$ mol &nbsp; (b) $N = nN_A = 0.802 \times 6.02\times10^{23} = 4.83\times10^{23}$

**Q2.** At constant volume $p \propto T$: $p_2 = 1.0\times10^5 \times \dfrac{450}{300} = 1.50\times10^{5}$ Pa. The temperatures must be in kelvin.

**Q3.** $c_{rms} = \sqrt{\dfrac{3k_BT}{m}} = \sqrt{\dfrac{3 \times 1.38\times10^{-23} \times 300}{28.0 \times 1.66\times10^{-27}}} = 517\ \mathrm{m\,s^{-1}}$, faster than the speed of sound in air.

**Q4.** $T = 300$ K, so $\overline{E_k} = \tfrac32 k_BT = 1.5 \times 1.38\times10^{-23} \times 300 = 6.21\times10^{-21}$ J. It is the same for every gas at this temperature.

**Q5.** $\tfrac12 m c_{rms}^2 = \tfrac32 k_BT \Rightarrow T = \dfrac{mc^2}{3k_B} = \dfrac{4.00 \times 1.66\times10^{-27} \times (1.12\times10^4)^2}{3 \times 1.38\times10^{-23}} = 2.01\times10^{4}$ K. Helium still escapes at much lower temperatures, because the fastest atoms in the Maxwell–Boltzmann tail move several times faster than $c_{rms}$.

**Q6.** $\eta = 200 \times \tfrac43\pi(0.03)^3 = 0.0226$, so the prediction is $1 + 4(0.0226) + 10(0.0226)^2 = 1.10$. The simulation below agrees to within a couple of per cent.

In [ ]:
for q, value in _answers.items():
    print(f"{q}: {value:#.3g}")
_g6 = kinetic.gas_box(n=200, radius=0.03, t_max=4, seed=5)
print(f"simulated pV/NkT = {_g6.pressure() * _g6.volume / (_g6.n_particles * _g6.temperature()):.3f}")

**Next:** in section 2, change `seed` to start from a different random arrangement. Does the final distribution change? Then try `dim=2` for discs in a square: the 2D distribution is $f(v) = \dfrac{mv}{k_BT}e^{-mv^2/2k_BT}$, and `kinetic.maxwell_boltzmann_speed(v, kT, dim=2)` draws it.

### Predict and explain: answers

**P1.** No. The speeds spread out into the lopsided Maxwell–Boltzmann shape: rising from zero, peaking a little below the starting speed, with a long tail of fast molecules. Each collision can share energy unevenly, with one molecule gaining what the other loses. The total is fixed, but how it is divided keeps changing, and the Maxwell–Boltzmann division is by far the most likely one.

**P2.** A little more: about 6% (the ratio is 1.059). The derivation treats molecules as points that fly freely from wall to wall. Real molecules take up space and bounce off each other, which effectively shortens the distance between wall hits, so they hit the walls more often. Hard-sphere theory predicts $1 + 4\eta = 1.054$ for the 1.3% of the box they fill.

**P3.** About 2.6 times ($1367/517$). At the same temperature every molecule has the same average kinetic energy, $\tfrac12m\overline{c^2} = \tfrac32k_BT$, so $c_{rms} \propto 1/\sqrt m$ and the ratio is $\sqrt{28/4} = 2.65$.